<a href="https://colab.research.google.com/github/mbaker21231/MicroII-Sandbox/blob/main/Hopenhayn2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Code to simulate Hopenhayn model

The model requires that one have a distribution of skills to work with, where $z$ is the measure of skills, typically deriving from a log-normal distribution. So, we will work with
$$
\ln z \sim N(\mu,\sigma)
$$

In fact, the usual assumption is that the skill distribution evolves over time according to:

$$
z_{t+1} = \rho z_t + \epsilon_t
$$

But usually, one wants to make this continuous distribution into a grid as it is easier to do dynamic programming. The usual method is the Tauchen method for doing this. Here is one way of doing it:

In [1]:
#Packages

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

I think it is a good idea to define all the parameters we are using in one place, and basically keep track of them on the basis of whether they are globals or locals, and experimental values.

Here are some of the parameters that we shall use, along with their values.

In [2]:
N     = 100          # Number of grid points to simulate the skill distribution
MU    = -.25         # Mean of skill distribution
RHO   = .85          # Serial correlation of skills over time
SIGMA = .40          # Variance of skill distribution
K_E   = 2            # Fixed costs of entry

# Global parameters

phi_c = 1            # Fixed labor costs of production
beta  = .05          # Discount factor
alpha = .5           # Cobb-Douglas production coefficient

Here is a simulator for the Tauchen distribution:

In [3]:
def tauchen(N, mu, rho, sigma, n_std=4):
    z = np.linspace(mu - n_std * sigma / np.sqrt(1 - rho**2),
                     mu + n_std * sigma / np.sqrt(1 - rho**2), N)
    step = (z[1] - z[0])
    P = np.zeros((N, N))

    for j in range(N):
        for k in range(N):
            if k == 0:
                P[j, k] = norm.cdf((z[k] - rho * z[j] + step / 2) / sigma)
            elif k == N-1:
                P[j, k] = 1 - norm.cdf((z[k] - rho * z[j] - step / 2) / sigma)
            else:
                P[j, k] = (norm.cdf((z[k] - rho * z[j] + step / 2) / sigma) -
                           norm.cdf((z[k] - rho * z[j] - step / 2) / sigma))

    return z, P


Note that we can also use this distribution to recover a cumulative unconditional distribution, which is useful for initial productivity draws:

In [4]:
def init_dist(N, mu, rho, sigma, n_std=4):

    tauch = tauchen(N, mu, rho, sigma, n_std)
    probs = np.sum(tauch[1], axis=0)/np.sum(tauch[1])
    vals = tauch[0]

    return vals, probs

## Some model details

Each firm has a flow profit function of the form:
$$
\pi(z) = \tilde z n^\alpha - Wn
$$

where $\tilde z$ is the (exponentiated) skill level $z$, $\tilde z=e^z$. Flow profits are acheived by choosing $n$, labor, to maximize the flow profits above:

$$
 \alpha \tilde z n^{\alpha -1}-W \quad \rightarrow\quad n^*(z,w) = \left(\frac{\alpha \tilde z}{W}\right)^\frac{1}{1-\alpha}
$$

Here is a function that returns, for a given wage and skill level, profits and labor demand:

In [5]:
def prof_lab(z, W):

  z_tilde = np.exp(z)
  n_sta   = ( alpha * z_tilde / W)**(1/(1-alpha))
  profs   = z_tilde*n_sta**alpha - W*alpha

  return profs, n_sta

## Present value of a firm

The present value of a firm follows a recursion that depends upon how the skill distribution evolves. To wit:

$$
V(z_t) = \max_{z^*} \left[0, \pi_t(z_t) + (1-\beta)\int_{z^*}^\infty V(\zeta)dF(\zeta|z_t)d\zeta\right]
$$

In the above, $z^*$ is the cutoff skill value, below which the firm exits the market, which determines in part the firm's current value.

The following bit of code essentially iterates the value function, taking into account that the firm's valuation changes as a result of possible changes in $z$, the skill level of the firm.

In [6]:
def val_fun(z, p, W, max_iter=3000, tol=1e-10, noisy=False):

  v = np.zeros((len(z), 1))

  for i in range(max_iter):

    profs = prof_lab(z, W)[0]
    profs = np.reshape(profs, (len(z), 1))
    vnew = np.maximum( 0, profs - phi_c + (1-beta)* p @ v)
    if np.max(abs(vnew-v)<tol):
      break

  if noisy:
    print("Iterations: ", i)
    print("Maximum value: ", np.max(vnew))
    print("Average value: ", np.mean(vnew))
    print("Minimum value: ", np.min(vnew))

  return vnew

## Computing an equilibrium wage - a quick digression

One thing we will want to do is compute an equilibrium wage; the idea is that there is a total of one unit of labor $(L=1)$ available to firms. so, what we ultimately want to do is, given a skill distribution, figure out how many firms are active (i.e., for which values of $z$ firms produce), determine their labor demands at a given wage, and then make sure aggregate labor demand adds up to the total one unit of labor.

Note that an assumption is that fixed costs of production are overhead labor as well, so we need to consider

In [9]:
Z, P = tauchen(N, MU, RHO, SIGMA)       # Skills and firms

Note that once we have computed the firms active at a given $W$, we need to a) weight labor demanded by relative frequency of skills, b) scale up the weighted labor demands by $M$, the mass of firms, c) include fixed costs, and then d) add everything up.

One complication: what is the distribution of skills? We will suppose just for this exercise that it is the long-run distribution as determined by our skill process. So, we will need a stationary distribution computer to do this. Here is one:

In [7]:
def stationary_distribution(P):
    n = P.shape[0]

    # Construct the matrix A and vector b
    A = np.vstack([P.T - np.eye(n), np.ones(n)])  # Add sum(π) = 1 constraint
    b = np.append(np.zeros(n), 1)  # Right-hand side

    # Solve using least squares
    pi = np.linalg.lstsq(A, b, rcond=None)[0]

    return pi

Now, we can compute the stationary distribution as follows:

In [10]:
PI = stationary_distribution(P)

Here is a loop that checks the difference between labor demand and labor supply for a given $W$ - the idea is that you can guess what $W$ is until labor demand is about 1. Give it a try!

In [25]:
W = 5.1                                   # Wage guess
M = 2                                   # Mass of firms
V = val_fun(Z, P, W, noisy=True)        # Value functions at skills

actives = (V > 0).astype(int)           # Indicator variable for active firms
pi, n = prof_lab(Z, W)                  # Profits, labor of all firms
n_active = n * actives                  # Labor demand of only those that are active

Ld = M * n_active * PI                  # Labor demand of active firms

np.sum(Ld), np.argmax(V > 0)


Iterations:  0
Maximum value:  22.297747311279426
Average value:  1.356755631452212
Minimum value:  0.0


(1.0294133074411236, 83)

In the above, we see that a wage a little bigger than 5 tends to work. Also, we see that only the firms with skill levels greater than position 83 are active!

## Dealing with Entrants

So, if I am understanding things correctly, we have to determine the measure of entrants. As Edmond has it, we have a law of motion for entrants as:

$$
\mu_{t+1}([0,z']) = \int F(z'|z)\mathbf{1}[z\geq z_t^*]\mu_t(dz)+m_{t+1}G(z')
$$

That looks difficult, but Edmond argues that this can be discretized to a grid with a certain number of elements, and we have:

$$
\mathbf{\mu_{t+1}} = \Psi_t\mathbf{\mu_t} + m_{t+1}\mathbf{g}
$$

If I'm reading everything correctly, the matrix $\Psi_t$ is basically the non-zero elements of the transition matrix. $m$ is a scalar, according to Edmonds. We note that what we want is the steady-state distribution of firms:

$$
\mathbf{\mu} = \Psi \mathbf{\mu} + m \mathbf{g}
$$

Let's try to solve this by iteration. Here, we work with an initial distribution consistent with our markov transition process, as determined by the function above. Let's just get this in the simplest fashion possible:

In [27]:
Z_init, P_init = init_dist(N, MU, RHO, SIGMA)

And really, the key thing is that the only thing that really matters for firms - the only way they interact  with one another - is through the wage. So, my guess is that we need to do the following loop:
1. Given an equilibrium wage, we compute which firms are active using profits and values.
2. Given a mass of entrants, and the fraction of active firms, we compute the long run distribution of firms.
3. Given the long run distribution of firms, we compute the equilibrium wage.

Repeat until convergence. We will use the same values we have been using throughout.

In [29]:
#### Iteration one.
V = val_fun(Z, P, W, noisy=True)        # Value functions at skills
beven = np.argmax(V > 1e-10)        # Index of first nonzero value

Iterations:  0
Maximum value:  22.297747311279426
Average value:  1.356755631452212
Minimum value:  0.0


In [31]:
# Iteration one continued... active firms
beven = np.argmax(V > 1e-10)        # Index of first nonzero value
pi, n = prof_lab(Z[beven], W)           # Profits, labor of active firms
print("Cutoff:", beven)

Cutoff: 83


So, the 83rd position is the first place where firms are active. It follows that all firms with z below z[83] are inactive, so we can say that these firms exit (consistent with what we decided above), although we made a mistake there in that we didn't also include the fixed costs of labor.

Anyways, we will fix that once this works. The idea is to make the inversion, we need a transition matrix that isn't degenerate, meaning only one that applies to active firms. We can

In [32]:
mask = (V > 0).astype(bool).flatten()
Psi = P[mask][:, mask]

Now, the entrants that are active are of mass $M$, multiplied by the initial distribution of skills, as follows:

In [35]:
G = (M * P_init)[mask]

Now, we can calculate the stationary distribution of firms, obtained by those that remain active that were already in the game, and those that entered. We have:

In [37]:
    I = np.eye(np.shape(Psi)[0])
    A = I - Psi

    Mact = np.linalg.solve(A, G)

In [38]:
Mact

array([0.03144691, 0.03253953, 0.03363421, 0.03470314, 0.03571974,
       0.03666096, 0.03750932, 0.03825457, 0.03889456, 0.03943534,
       0.03989042, 0.04027925, 0.04062515, 0.04095304, 0.041287  ,
       0.04164829, 0.05348606])

In [39]:
np.sum(Mact)

0.6569675039275795